# Comparing the two blends

* **Equal weights:** the method that `forecast_evaluation.ipynb` picks in section 7 (currently `mean of all`)
* **Per-hour weights:** `hourly_weights_blend.ipynb` (weights per forecast hour from the RMSE)

This notebook does not copy any code. It runs both notebooks (without plots, files or downloads), takes their blending
functions, and scores both on **the same test forecasts**, which neither method was fitted on.

Change a setting in one of the two notebooks (e.g. `EXCLUDE_MODELS`, `WINDOW`), save it, and run this notebook again.

In [7]:
import io, json, contextlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import IPython.display

DOWNLOAD = False      # True = let the notebooks fetch new forecast files from GitHub first
MAX_HORIZON = 96      # compare forecast hours 1-96 (4 days)

In [8]:
def run_notebook(path, stop_at, skip=(), overrides=None):
    """Run the code cells of a notebook until the first cell containing `stop_at`.
    Returns all its variables. Plots, saved figures, tables and printing are switched off."""
    ns = {'__name__': '__notebook__'}
    sources = [''.join(c['source']) for c in json.load(open(path, encoding='utf-8'))['cells']
               if c['cell_type'] == 'code']
    show, savefig, disp = plt.show, plt.savefig, IPython.display.display
    plt.show = lambda *a, **k: plt.close('all')
    plt.savefig = lambda *a, **k: None
    IPython.display.display = lambda *a, **k: None
    ns['display'] = IPython.display.display
    try:
        with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for src in sources:
                if stop_at in src:
                    break
                if any(s in src for s in skip):
                    continue
                exec(compile(src, path, 'exec'), ns)
                ns.update(overrides or {})
    finally:
        plt.show, plt.savefig, IPython.display.display = show, savefig, disp
        plt.close('all')
    return ns

def utc(t):
    t = pd.Timestamp(t)
    return t.tz_localize('UTC') if t.tzinfo is None else t

print('Running forecast_evaluation.ipynb ...')
A = run_notebook('forecast_evaluation.ipynb', stop_at='group_number',
                 skip=[] if DOWNLOAD else ['download_forecasts('])
print('Running hourly_weights_blend.ipynb ...')
B = run_notebook('hourly_weights_blend.ipynb', stop_at='methods_to_try',
                 overrides={'DOWNLOAD_NEW_FILES': DOWNLOAD})
print('Done.')

Running forecast_evaluation.ipynb ...
Running hourly_weights_blend.ipynb ...
Done.


## Same test forecasts for both
Both notebooks use the newest 30 % of the forecast files as test files. Here only the files that are test files **in both** are used, and only daytime hours with a measurement, horizons 1–96 h.

In [9]:
split = max(utc(A['split_time']), utc(B['split_time']))
t = B['table']
issued = t.index.get_level_values('issue_time')
test = t[(issued >= split) & (t['horizon'] >= 1) & (t['horizon'] <= MAX_HORIZON)].copy()

# Equal weights: the method forecast_evaluation picked, with weights fitted on its training files
weights_A = A['methods'][A['best_method']](A['train'])
test['equal_weights'] = A['apply_blend'](test, weights_A)

# Per-hour weights: fitted on the training files of hourly_weights_blend
test['per_hour_weights'] = B['apply_weights'](test, B['w_train'])

# Per-hour weights, walk-forward (weights re-made before every forecast from what was measured then)
if 'wf' in B:
    test = test.join(B['wf'][['blend_walk_forward']].rename(columns={'blend_walk_forward': 'per_hour_walk_forward'}))

best_single = A['best_single']
compare = ['equal_weights', 'per_hour_weights'] + (['per_hour_walk_forward'] if 'per_hour_walk_forward' in test else []) + [best_single]
test = test.dropna(subset=compare + ['measured'])
test['day'] = (test['horizon'] - 1) // 24 + 1

print('Test files used:', test.index.get_level_values('issue_time').nunique(),
      '  issued from', split, '  daytime hours:', len(test))
print()
print('Equal weights     -', A['best_method'], 'over', A['blend_models'])
used_B = [m for m in B['models'] if B['w_train'][m].sum() > 0]
print('Per-hour weights  -', B['WEIGHT_METHOD'], f"±{B['WINDOW']} h over", used_B)
if 'dmi_seamless' in A['blend_models'] or 'dmi_seamless' in used_B:
    print('NOTE: dmi_seamless is used - it is a copy of Harmonie (<=54 h) and ecmwf_ifs (>60 h).')

Test files used: 82   issued from 2026-11-18 00:00:00+00:00   daytime hours: 1095

Equal weights     - 0.3 ecmwf_aifs025_single + 0.7 ecmwf_ifs over ['ecmwf_ifs025', 'ecmwf_aifs025_single', 'gfs_global', 'ecmwf_ifs']
Per-hour weights  - inverse_mse ±3 h over ['dmi_harmonie_arome_europe', 'metno_nordic', 'ecmwf_ifs025', 'ecmwf_aifs025_single', 'gfs_global', 'ecmwf_ifs']


## Results

In [10]:
rmse = lambda e: np.sqrt((e ** 2).mean())
rows = {}
for c in compare:
    e = test[c] - test['measured']
    row = {f'RMSE day {d}': rmse(e[test['day'] == d]) for d in range(1, 5)}
    row.update({'RMSE 1-96 h': rmse(e), 'bias': e.mean(), 'MAE': e.abs().mean()})
    rows[c] = row
summary = pd.DataFrame(rows).T
summary.index.name = 'forecast'
display(summary.round(1))

# File by file: which blend is better for each test forecast?
per_file = test.groupby(level='issue_time').apply(
    lambda g: pd.Series({c: rmse(g[c] - g['measured']) for c in ['equal_weights', 'per_hour_weights']}))
wins = (per_file['per_hour_weights'] < per_file['equal_weights']).mean()
diff = 100 * (1 - summary.loc['per_hour_weights', 'RMSE 1-96 h'] / summary.loc['equal_weights', 'RMSE 1-96 h'])
print(f'Per-hour weights vs. equal weights: {diff:+.1f} % lower RMSE (negative = worse)')
print(f'Per-hour weights are better in {100 * wins:.0f} % of the {len(per_file)} test files')

summary.round(2).to_csv('blend_comparison.csv')

,RMSE day 1,RMSE day 2,RMSE day 3,RMSE day 4,RMSE 1-96 h,bias,MAE
forecast,,,,,,,
equal_weights,24.0,24.4,24.9,7.4,24.0,1.3,17.1
per_hour_weights,33.2,31.4,28.1,10.8,31.5,3.9,23.5
per_hour_walk_forward,32.9,31.8,26.5,11.2,31.3,4.2,23.3
ecmwf_ifs,25.0,24.0,29.1,6.1,25.1,0.7,18.0


Per-hour weights vs. equal weights: -31.3 % lower RMSE (negative = worse)
Per-hour weights are better in 4 % of the 82 test files


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
style = {'equal_weights': dict(color='grey', linewidth=2.5, linestyle='--'),
         'per_hour_weights': dict(color='black', linewidth=2.5),
         'per_hour_walk_forward': dict(color='black', linewidth=1.2, linestyle=':'),
         best_single: dict(color='tab:blue', linewidth=1.2)}

# Left: RMSE per forecast day
x = np.arange(1, 5)
width = 0.8 / len(compare)
for i, c in enumerate(compare):
    axes[0].bar(x - 0.4 + width / 2 + i * width, summary.loc[c, [f'RMSE day {d}' for d in x]], width,
                label=c, color=style[c]['color'], alpha=1 if 'per_hour' in c else 0.6,
                hatch='//' if c == 'per_hour_walk_forward' else None, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'day {d}' for d in x])
axes[0].set_ylabel('RMSE [W/m²]')
axes[0].set_title('Test forecasts: RMSE per forecast day')
axes[0].legend(fontsize=8)

# Right: RMSE per forecast hour, smoothed over ±3 h
for c in compare:
    e2 = ((test[c] - test['measured']) ** 2).groupby(test['horizon'])
    smooth = np.sqrt(e2.sum().rolling(7, center=True, min_periods=1).sum() /
                     e2.size().rolling(7, center=True, min_periods=1).sum())
    axes[1].plot(smooth.index, smooth.values, label=c, **style[c])
axes[1].set_xlabel('Forecast horizon [hours]')
axes[1].set_ylabel('RMSE [W/m²] (smoothed ±3 h)')
axes[1].set_title('Test forecasts: RMSE per forecast hour')
for d in (24, 48, 72):
    axes[1].axvline(d, color='grey', linewidth=0.5, linestyle=':')

plt.tight_layout()
plt.savefig('blend_comparison.png', dpi=200, bbox_inches='tight')
plt.show()